# 🏨 AGODA Price Crawler

Chạy lần lượt các cell từ trên xuống: **① Cấu hình → ② Đọc input → ③ Crawl → ④ Xem kết quả**.

**Đổi nguồn input:** sửa `INPUT_MODE` ở cell ① — `"gsheet"` (Google Sheet online) hoặc `"offline"` (file CSV/XLSX trên máy).

**Format file offline:** chỉ cần **3 cột đầu theo đúng thứ tự** `hotel_name, hotel_url, room_type` (tên cột không quan trọng, chỉ cần đúng thứ tự). Có file mẫu ở `input/TEMPLATE_hotels.csv`.

**Output** nằm trong `results/agoda/`:
- `FINAL_<YYYYMMDD>.csv` — kết quả cuối
- `TEMP_agoda.csv` — checkpoint: lỡ tắt giữa chừng, chạy lại cell ③ sẽ tự resume phần chưa xong

In [1]:
# ════════════════ ① CẤU HÌNH ════════════════

# ── Nguồn input: "gsheet" (online) hoặc "offline" (file trên máy) ──
INPUT_MODE = "gsheet"

# Dùng khi INPUT_MODE = "gsheet" (gid của tab được tự lấy từ URL)
GSHEET_URL = "https://docs.google.com/spreadsheets/d/1EauwTNOMVMBT_CUHwf4EtOz1nsZAjG2SJ_LEh199N0Q/edit?gid=1289817800#gid=1289817800"

# Dùng khi INPUT_MODE = "offline" — đường dẫn tuyệt đối, hoặc tương đối so với 31.crawl-tool
# ⚠️ File phải có 3 cột đầu là (tên KS, URL, loại phòng) — file "TEMP_*" là checkpoint OUTPUT, không phải input!
OFFLINE_FILE = "input/v"

# ── Tham số crawl ──
WEEKS      = 6      # số tuần cần crawl
MAX_HOTELS = 0      # 0 = crawl tất cả; đặt 5 để test nhanh 5 khách sạn đầu
SHARD      = ""     # "" = không chia; "1/3" = chạy phần 1 trong 3 phần (chạy lần lượt 1/3, 2/3, 3/3)

In [2]:
# ════════════════ ② ĐỌC INPUT ════════════════
import os, sys

if "ROOT" not in globals():                    # giữ nguyên ROOT khi chạy lại cell
    ROOT = os.path.abspath("")                 # .../31.crawl-tool (nơi đặt notebook này)
assert os.path.isdir(os.path.join(ROOT, "crawler")), (
    f"Không tìm thấy package `crawler` trong {ROOT} — hãy mở notebook từ thư mục 31.crawl-tool")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import crawler
from crawler.hotels_io import read_hotels

if INPUT_MODE == "gsheet":
    INPUT = GSHEET_URL
    print("📡 Input: Google Sheet online")
else:
    INPUT = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
    assert os.path.exists(INPUT), f"Không tìm thấy file: {INPUT}"
    print(f"📁 Input: file offline — {INPUT}")

hotels = read_hotels(INPUT)
print(f"✅ Đọc được {len(hotels)} khách sạn. 5 dòng đầu:")
for name, url, room in hotels[:5]:
    print(f"   • {name} — {room}")

📡 Input: Google Sheet online
✅ Đọc được 90 khách sạn. 5 dòng đầu:
   • Harvest Day Hoi An - Hotel — Standard Double with Garden View
   • Little Oasis - Hotel — Little Oasis Deluxe
   • Grand Sunrise Palace Hội An - Hotel — Phòng Loại Sang Giường Đôi (Deluxe Double Room)
   • Reu Boutique Hotel - Hotel — Superior 2 giường hoặc giường đôi (Superior Twin or Double)
   • Maison Vy - Hotel — Giường Lớn Cao Cấp (Superior King)


In [3]:
# (TÙY CHỌN) Tải Google Sheet về file offline — lần sau chỉ cần đổi INPUT_MODE = "offline"
import pandas as pd
from crawler.hotels_io import _gsheet_url

os.makedirs(os.path.join(ROOT, "input"), exist_ok=True)
dest = os.path.join(ROOT, "input", "agoda_hotels.csv")
pd.read_csv(_gsheet_url(GSHEET_URL)).to_csv(dest, index=False, encoding="utf-8-sig")
print(f"💾 Đã lưu bản offline: {dest}")

💾 Đã lưu bản offline: /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/input/agoda_hotels.csv


In [ ]:
# ════════════════ ③ CRAWL ════════════════
OUTDIR = os.path.join(ROOT, "results", "agoda")
os.makedirs(OUTDIR, exist_ok=True)
os.chdir(OUTDIR)                     # output (FINAL_*.csv, TEMP_agoda.csv) nằm ở đây

kwargs = dict(
    site="agoda",                    # direct replay (nhanh) + Camoufox warm
    input=INPUT,
    weeks=WEEKS,
)
if MAX_HOTELS:
    kwargs["max"] = MAX_HOTELS
if SHARD:
    kwargs["shard"] = SHARD

await crawler.arun(**kwargs)         # notebook cho phép await trực tiếp

🚀 AGODA crawl | 90 hotels × 6w | direct+fallback | engine=camoufox | W1=2026-07-14

🏨 1/90 Harvest Day Hoi An - Hotel | Standard Double with Garden View | need weeks [1, 2, 3, 4, 5, 6]
   ✅ 6/6 priced (direct 6, fallback 0) | pace limit=3

🏨 2/90 Little Oasis - Hotel | Little Oasis Deluxe | need weeks [1, 2, 3, 4, 5, 6]
   ✅ 6/6 priced (direct 6, fallback 0) | pace limit=4

🏨 3/90 Grand Sunrise Palace Hội An - Hotel | Phòng Loại Sang Giường Đôi (Deluxe Double Room) | need weeks [1, 2, 3, 4, 5, 6]
   ✅ 6/6 priced (direct 6, fallback 0) | pace limit=4

🏨 4/90 Reu Boutique Hotel - Hotel | Superior 2 giường hoặc giường đôi (Superior Twin or Double) | need weeks [1, 2, 3, 4, 5, 6]
   ✅ 6/6 priced (direct 6, fallback 0) | pace limit=5

🏨 5/90 Maison Vy - Hotel | Giường Lớn Cao Cấp (Superior King) | need weeks [1, 2, 3, 4, 5, 6]
   ✅ 6/6 priced (direct 6, fallback 0) | pace limit=5

🏨 6/90 Sen Village Hội An (Sen Village Hoi An) - Hotel | Phòng Loại Sang (Deluxe Room) | need weeks [1, 2, 3, 4

In [ ]:
# ════════════════ ④ XEM KẾT QUẢ ════════════════
import glob
import pandas as pd

OUTDIR = os.path.join(ROOT, "results", "agoda")
files = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_*.csv")))
assert files, "Chưa có file FINAL nào — hãy chạy cell ③ trước."
latest = files[-1]
df = pd.read_csv(latest)
print(f"📄 {latest} — {len(df)} dòng")
df.head(20)